# Session 14 · Case study: embedding features versus TF-IDF, and nearest-neighbour search

**Block 2 practice.** Compare frozen sentence embeddings with TF-IDF as classification features on the same subsample, and build a nearest-neighbour search over reviews.

Needs `sentence-transformers` (first run downloads `all-MiniLM-L6-v2`, about 90 MB). Encoding 10,000 reviews takes a few seconds with a GPU and about a minute on a CPU; the vectors are cached in `embeddings_cache/` next to this notebook.

Theory: [02-applications-of-embeddings.md](../theory/02-applications-of-embeddings.md).

*Author: course team, licence CC-BY-4.0.*

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"
reviews = pd.read_parquet(DATA / "train_sample.parquet")
print(reviews.shape)

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
sub = reviews.groupby("label").sample(frac=0.2, random_state=0)          # 10,000 reviews
texts, y = sub["title"] + ". " + sub["text"], sub["label"]
X_tr, X_va, y_tr, y_va = train_test_split(texts, y, test_size=0.25, stratify=y, random_state=0)
model = SentenceTransformer(MODEL_NAME)

cache = Path("embeddings_cache")
cache.mkdir(exist_ok=True)


def embed(name, docs):
    """Encode once, then load from disk; the file name records model and split."""
    path = cache / f"{MODEL_NAME.split('/')[-1]}_{name}.npy"
    if path.exists():
        return np.load(path)
    E = model.encode(docs.tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    np.save(path, E)
    return E


E_tr, E_va = embed("train", X_tr), embed("valid", X_va)
print(E_tr.shape, E_va.shape)

## 1. Three representations, one classifier

In [ ]:
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True).fit(X_tr)
T_tr, T_va = vec.transform(X_tr), vec.transform(X_va)
features = {
    "tf-idf": (T_tr, T_va, 4),
    "embeddings": (E_tr, E_va, 1),
    "both": (hstack([T_tr, csr_matrix(E_tr)]).tocsr(), hstack([T_va, csr_matrix(E_va)]).tocsr(), 4),
}
rows, preds = [], {}
for name, (A_tr, A_va, C) in features.items():
    clf = LogisticRegression(C=C, class_weight="balanced", max_iter=3000).fit(A_tr, y_tr)
    preds[name] = clf.predict(A_va)
    per_class = f1_score(y_va, preds[name], average=None, labels=["neg", "neu", "pos"])
    rows.append({"features": name, "macro_F1": f1_score(y_va, preds[name], average="macro"),
                 "F1_neg": per_class[0], "F1_neu": per_class[1], "F1_pos": per_class[2]})
pd.DataFrame(rows).round(3)

## 2. Learning curve: does the number of labels change the winner?

In [ ]:
curve = []
for n in [200, 500, 1000, 3000, len(y_tr)]:
    idx = y_tr.groupby(y_tr).sample(frac=n / len(y_tr), random_state=0).index if n < len(y_tr) else y_tr.index
    pos = X_tr.index.get_indexer(idx)
    emb = LogisticRegression(C=10 if n < 3000 else 1, class_weight="balanced", max_iter=3000)
    emb.fit(E_tr[pos], y_tr.loc[idx])
    v = TfidfVectorizer(ngram_range=(1, 2), min_df=1 if n < 3000 else 3, sublinear_tf=True)
    tf = LogisticRegression(C=4, class_weight="balanced", max_iter=3000).fit(v.fit_transform(X_tr.loc[idx]), y_tr.loc[idx])
    curve.append({"n_train": len(idx),
                  "embeddings": f1_score(y_va, emb.predict(E_va), average="macro"),
                  "tf-idf": f1_score(y_va, tf.predict(v.transform(X_va)), average="macro")})
curve = pd.DataFrame(curve).set_index("n_train")
ax = curve.plot(marker="o", logx=True, ylabel="macro-F1 (validation)", title="Learning curve")
curve.round(3)

**Question.** Where do the two curves cross? What would you recommend for a new label set (for example "complaint about packaging: yes/no") with 300 labelled reviews?

## 3. Nearest-neighbour search over reviews

In [ ]:
corpus = X_va.reset_index(drop=True)
kw = TfidfVectorizer().fit(corpus)
K = kw.transform(corpus)


def search(query, k=5):
    q = model.encode(query, normalize_embeddings=True)
    emb_scores = E_va @ q
    kw_scores = (K @ kw.transform([query]).T).toarray().ravel()
    out = []
    for method, scores in [("embedding", emb_scores), ("tf-idf", kw_scores)]:
        for rank, i in enumerate(np.argsort(-scores)[:k], start=1):
            out.append({"method": method, "rank": rank, "score": round(float(scores[i]), 2),
                        "label": y_va.iloc[i], "review": corpus[i][:110]})
    return pd.DataFrame(out)


pd.set_option("display.max_colwidth", 120)
search("stopped working after a week")

In [ ]:
search("smells terrible")

Look at the labels of the hits for "smells terrible": the embedding search finds reviews about smell, positive and negative. Embeddings encode the topic more strongly than the opinion.

## 4. Nearest neighbours of a review

The same index answers "which reviews are most similar to this one?", for example to find duplicates or recurring complaints.

In [ ]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=4, metric="cosine").fit(E_va)
dist, ind = nn.kneighbors(E_va[[10]])
for d, i in zip(dist[0], ind[0]):
    print(f"{1 - d:.2f}  {y_va.iloc[i]}  {corpus[i][:100]}")

## Your turn

1. Write three queries of your own. For each, decide which method gives the more useful top-5, and why.
2. Find a query where TF-IDF is clearly better (hint: product names, numbers or negations).
3. Optional: cluster `E_va` with k-means (k = 6) and describe each cluster by its most distinctive TF-IDF words, as on the theory page. Which cluster has the highest share of negative reviews?
4. Optional: repeat section 1 with a multilingual model (`paraphrase-multilingual-MiniLM-L12-v2`, about 0.5 GB) and compare.

In [ ]:
# your code here